# OE2: comprobación reproducible en Google Colab

**Objetivo oficial (literal).** Determinar estadísticamente el desempeño propio de la propuesta basada en inteligencia de enjambre, considerando como indicadores el tiempo de convergencia, la latencia y el throughput bajo condiciones experimentales controladas.

**Qué comprueba este cuaderno.** Comprueba las dos mediciones del objetivo en un solo flujo: **componente A** (tiempo de convergencia, latencia y throughput en proceso, 5 000 ciclos) y **componente B** (latencia, throughput y tiempo de convergencia por HTTP, 120 corridas válidas). Para B ejecuta dos etapas en orden: B1, recálculo de los indicadores por corrida desde cada petición registrada; B2, análisis pre-registrado y exploratorio desde los indicadores por corrida. La conclusión final se calcula para ambos componentes.

**Datos de entrada.** Componente A: observaciones por ciclo (`raw_observations.csv`), tiempos de pared por lote, monitor y compuertas de reposo. Componente B: registros por petición, `runs.jsonl` (136 registros finales), `intentos.jsonl`, monitores de los dos segmentos y planes. Scripts de análisis sellados y resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Ejecute las celdas en orden en un entorno nuevo de Colab; la carga de datos pide subir los archivos indicados.

**Contenido (diez pasos).** 1 descripción; 2 datos de entrada; 3 verificación de archivos y estructura; 4 dependencias; 5 ejecución del análisis; 6 reconstrucción de los indicadores; 7 comparación con los resultados oficiales; 8 diferencias numéricas; 9 verificación de las salidas finales; 10 conclusión de reproducibilidad (calculada por la última celda).

In [ ]:
import sys, platform, os
print('Python', sys.version); print(platform.platform())
!pip -q install numpy scipy pandas statsmodels
import numpy, scipy, pandas, statsmodels; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__, '| statsmodels', statsmodels.__version__)
print('CPU', os.cpu_count())

In [ ]:
import zipfile, glob
PARTES = 2
NECESARIOS = ['oe2a_colab_paquete.zip', 'oe2b_colab_paquete.zip']
partes = sorted(glob.glob('oe2b_http_p*.zip'))
faltan = [z for z in NECESARIOS if not os.path.exists(z)]
if len(partes) < PARTES or faltan:
    from google.colab import files
    print('Suba los archivos:', NECESARIOS, 'y las', PARTES, 'partes oe2b_http_p1.zip ... oe2b_http_p'+str(PARTES)+'.zip')
    files.upload()
    partes = sorted(glob.glob('oe2b_http_p*.zip'))
assert len(partes) == PARTES, ('partes encontradas:', partes)
assert all(os.path.exists(z) for z in NECESARIOS), 'faltan paquetes de analisis'
for z in NECESARIOS + partes: zipfile.ZipFile(z).extractall('.')
print(len(partes), 'partes http extraidas + paquetes de analisis', NECESARIOS, '|', sorted(os.listdir('http')))

## Componente A: análisis desde las observaciones por ciclo

In [ ]:
import hashlib
base='oe2a/campana_01'; bad=[]; checked=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,n=l.split(None,1); n=n.strip().lstrip('*'); p=os.path.join(base,n)
    if os.path.exists(p):
        checked.append(n)
        if hashlib.sha256(open(p,'rb').read()).hexdigest()!=h: bad.append(n)
print('verificados:', checked); print('hash distinto:', bad); assert not bad

In [ ]:
!python oe2a/analisis_oe2a.py oe2a/campana_01/raw_observations.csv oe2a/campana_01/lotes_tiempo_pared.json oe2a/salida_colab oe2a/campana_01_monitor.csv oe2a/gate_inicio_campana01.json oe2a/gate_cierre_campana01.json > oe2a/log_analisis.txt 2>&1
print(open('oe2a/log_analisis.txt').read()[-400:])

In [ ]:
import json, math
a=json.load(open('oe2a/salida_colab/resultados_oe2a.json')); b=json.load(open('oe2a/esperado_resultados_oe2a_local.json'))
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
A=dict(hoja(a)); B=dict(hoja(b))
print('mismas claves:', set(A)==set(B), '| hojas:', len(A))
mx=0; peor=None; dist=[]
for k in A:
    va, vb = A[k], B.get(k)
    if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool):
        dif=abs(va-vb); rel=dif/max(abs(vb),1e-12)
        if dif>mx: mx=dif; peor=(k,va,vb)
        if rel>1e-9 and dif>1e-9: dist.append((k,va,vb))
    elif va!=vb: dist.append((k,va,vb))
print('diferencia absoluta maxima:', mx, peor)
print('hojas distintas (rel>1e-9):', len(dist)); print(dist[:15])
print('T_conv media geometrica (ms): Colab', a['tconv_I02']['media_geometrica_ms'], '| local', b['tconv_I02']['media_geometrica_ms'])
print('IC95 t:', a['tconv_I02']['ic95_t_ms'], b['tconv_I02']['ic95_t_ms'])
print('IC95 bootstrap:', a['tconv_I02']['ic95_bootstrap_percentil_ms'], b['tconv_I02']['ic95_bootstrap_percentil_ms'])
print('P95 latencia en proceso:', a['latencia_en_proceso']['percentiles_ms_ciclos']['95'], b['latencia_en_proceso']['percentiles_ms_ciclos']['95'])
print('X_proc entre lotes:', a['throughput_en_proceso']['entre_lotes']['media'], b['throughput_en_proceso']['entre_lotes']['media'])
print('CR:', a['tasa_convergencia_I03']['CR'], a['tasa_convergencia_I03']['ic95_wilson'])
print('instrumentacion cumple:', a['instrumentacion']['cumple'], b['instrumentacion']['cumple'])

## Componente B, etapa 1 (B1): indicadores por corrida desde las peticiones registradas

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe2b.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
import json
ANON={a['archivo']:a['sha256_anonimizado'] for a in json.load(open('http/oe2b/campana_01/anonimizacion_oe2b.json',encoding='utf8'))['archivos']}
base='http/oe2b/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=ANON.get(nm,h): mal.append(nm)
print('oe2b: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt (los archivos anonimizados, contra su suma anonimizada):', ok, '| distintos:', mal); assert not mal

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe2b',):
    !python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe2b',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

In [ ]:
import pandas as pd, math
def runs_de(k): return {json.loads(l)['label']: json.loads(l) for l in open(f'http/{k}/campana_01/runs.jsonl', encoding='utf8') if l.strip()}
# OE2-B: tabla por carga del informe (media aritmética del P95, throughput y T_conv por corrida; tasa de error acumulada) desde los valores RECALCULADOS
rc=pd.read_csv('http/out/recomputo_oe2b.csv'); rc=rc[rc['valida_final']==True].copy(); rr=runs_de('oe2b')
rc['carga']=rc['label'].map(lambda s: rr[s]['load'])
g=rc.groupby('carga').agg(n=('label','size'), p95=('latency_p95_ms_recalculado','mean'), thr=('throughput_rps_recalculado','mean'), tconv=('t_conv_ms_mean_recalculado','mean'), req=('requests_total_recalculado','sum'), err=('errors_recalculado','sum')).reset_index()
g['error_tasa']=g['err']/g['req']
e=pd.read_csv('http/oe2b/esperado_resumen_por_carga.csv')
m=g.merge(e,on='carga')
d=lambda x,y: float((m[x]-m[y]).abs().max())
print('OE2-B por carga (n=20 corridas válidas): dif abs máx desde HTTP crudo frente a la tabla del análisis | P95 media:', d('p95','p95_ms_media'), '| throughput media:', d('thr','thr_media'), '| T_conv media:', d('tconv','tconv_ms_media'), '| tasa de error:', d('error_tasa_x','error_tasa_y'))
print(m[['carga','n_x','p95','p95_ms_media','thr','thr_media']].to_string(index=False))


## Componente B, etapa 2 (B2): análisis pre-registrado y exploratorio desde los indicadores por corrida

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('oe2b/hashes_paquete_oe2b.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe2b',nm); n+=1
    if sha(p)!=h: bad.append(nm)
print('archivos del paquete verificados contra el hash local:', n, '| distintos:', bad); assert not bad
import json
ANON={a['archivo']:a['sha256_anonimizado'] for a in json.load(open('oe2b/campana_01/anonimizacion_oe2b.json',encoding='utf8'))['archivos']}
bad2=[]; ok2=[]
for l in open('oe2b/campana_01/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe2b/campana_01',nm)
    if os.path.exists(p):
        ok2.append(nm)
        if sha(p)!=ANON.get(nm,h): bad2.append(nm)
print('contra el SHA256SUMS.txt de la campana (los seis archivos anonimizados contra su suma anonimizada), archivos presentes:', ok2, '| distintos:', bad2); assert not bad2

## Monitor de recursos de los dos segmentos
Se concatena el monitor del segmento 1 con el del segmento 2 (sin su encabezado), igual que en el análisis local, y se compara con el archivo derivado local.

In [ ]:
seg1=open('oe2b/monitor_segmento1.csv','rb').read()
seg2=open('oe2b/monitor_segmento2.csv','rb').read().split(b'\n',1)[1]
os.makedirs('oe2b/salida_colab',exist_ok=True)
open('oe2b/salida_colab/monitor_concatenado.csv','wb').write(seg1+seg2)
print('monitor concatenado igual al derivado local (SHA-256):', sha('oe2b/salida_colab/monitor_concatenado.csv')==sha('oe2b/esperado_local/monitor_concatenado_derivado.csv'))

## Conteo de corridas (120 válidas, 16 no válidas reemplazadas, intentos)

In [ ]:
import json, pandas as pd
R=[json.loads(l) for l in open('oe2b/campana_01/runs.jsonl',encoding='utf8') if l.strip()]
I=[json.loads(l) for l in open('oe2b/campana_01/intentos.jsonl',encoding='utf8') if l.strip()]
r=pd.DataFrame(R); print('registros finales:', len(r), '| validos:', int((r.valid==True).sum()), '| no validos:', int((r.valid!=True).sum()), '| intentos totales:', len(I))
print(r[r.valid==True].groupby('load').size().to_dict())

## 1. Análisis PRE-REGISTRADO (`analisis_oe2b.py`, mismo script y mismos argumentos que en local)

In [ ]:
!python oe2b/analisis_oe2b.py oe2b/campana_01/runs.jsonl oe2b/salida_colab/preregistrado oe2b/salida_colab/monitor_concatenado.csv > oe2b/salida_colab/log_preregistrado.txt 2>&1
print(open('oe2b/salida_colab/log_preregistrado.txt').read()[-700:])

## 2. Análisis EXPLORATORIO (NO pre-registrado; estratificación posterior por Zoom)

In [ ]:
!python oe2b/analisis_exploratorio_oe2b.py oe2b/campana_01 oe2b/salida_colab/monitor_concatenado.csv oe2b/salida_colab/exploratorio > oe2b/salida_colab/log_exploratorio.txt 2>&1
print(open('oe2b/salida_colab/log_exploratorio.txt').read()[-900:])

## 3. Comparación Colab vs local (diferencias absolutas y relativas)

In [ ]:
import json, math
import pandas as pd
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d in dist[:20]: print('   DIFERENCIA', d)
    return dict(etiqueta=etiqueta, mismas_claves=mismas, hojas=len(A), numericas=n_num, dif_abs_max=mx, dif_rel_max=mxrel, hojas_con_dif=n_dist, hojas_dif_gt_1e9=len(dist))
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
    return dict(etiqueta=etiqueta, misma_forma=ok, filas=a.shape[0], dif_abs_max=mx, dif_rel_max=mxrel, celdas_distintas=nd)

In [ ]:
res=[]
res.append(comparar_json('oe2b/salida_colab/preregistrado/resultados_oe2b.json','oe2b/esperado_local/resultados_oe2b.json','OE2-B pre-registrado: resultados_oe2b.json'))
res.append(comparar_csv('oe2b/salida_colab/preregistrado/resumen_por_carga.csv','oe2b/esperado_local/resumen_por_carga.csv','OE2-B pre-registrado: resumen_por_carga.csv'))
res.append(comparar_json('oe2b/salida_colab/exploratorio/exploratorio_oe2b.json','oe2b/esperado_local/exploratorio_oe2b.json','OE2-B EXPLORATORIO: exploratorio_oe2b.json'))
res.append(comparar_csv('oe2b/salida_colab/exploratorio/estratos_por_carga.csv','oe2b/esperado_local/estratos_por_carga.csv','OE2-B EXPLORATORIO: estratos_por_carga.csv'))
pd.DataFrame(res).to_csv('oe2b/salida_colab/comparacion_colab_vs_local.csv',index=False)
print(pd.DataFrame(res)[['etiqueta','dif_abs_max','dif_rel_max']].to_string())

## 4. Indicadores principales (Colab | local) y conclusión por requisito

In [ ]:
a=json.load(open('oe2b/salida_colab/preregistrado/resultados_oe2b.json',encoding='utf8')); b=json.load(open('oe2b/esperado_local/resultados_oe2b.json',encoding='utf8'))
print('validas', a['n_corridas_validas'], b['n_corridas_validas'], '| no validas', a['n_no_validas'], b['n_no_validas'], '| intentos extra', a['intentos_extra'], b['intentos_extra'], '| interferidas', a['interferidas'], b['interferidas'], '| generador saturado', a['generador_saturado'], b['generador_saturado'])
rows=[]
for L in ['1','5','10','25','50','100']:
    x=a['resultados']['por_carga'][L]; y=b['resultados']['por_carga'][L]; e=a['resultados']['error'][L]; f=b['resultados']['error'][L]
    rows.append(dict(carga=L, n=x['n_corridas'], p95_colab=round(x['p95_ms']['media'],3), p95_local=round(y['p95_ms']['media'],3), ic_p95_colab=[round(x['p95_ms']['ic95_t_log']['lo'],1),round(x['p95_ms']['ic95_t_log']['hi'],1)], thr_colab=round(x['throughput_rps']['media'],4), thr_local=round(y['throughput_rps']['media'],4), tconv_colab=round(x['t_conv_ms']['media'],3), tconv_local=round(y['t_conv_ms']['media'],3), err_colab=round(e['tasa'],6), err_local=round(f['tasa'],6)))
print(pd.DataFrame(rows).to_string())
for k in ('contrastes_p95','contrastes_throughput'):
    print(k); print(pd.DataFrame({L:{'p_holm_colab':a['resultados'][k][L]['p_holm'],'p_holm_local':b['resultados'][k][L]['p_holm'],'cumple_colab':[v for kk,v in a['resultados'][k][L].items() if kk.startswith('cumple')][0],'cumple_local':[v for kk,v in b['resultados'][k][L].items() if kk.startswith('cumple')][0]} for L in a['resultados'][k]}).T.to_string())
print('deriva por orden (Spearman):', a['resultados']['estabilidad']['deriva_orden_log_p95'], '|', b['resultados']['estabilidad']['deriva_orden_log_p95'])
print('efecto de bloque (Kruskal):', a['resultados']['estabilidad']['efecto_bloque_log_p95_kruskal'], '|', b['resultados']['estabilidad']['efecto_bloque_log_p95_kruskal'])
print('sensibilidad sin interferidas:', a['sensibilidad_sin_interferidas_ni_generador_saturado'])
ea=json.load(open('oe2b/salida_colab/exploratorio/exploratorio_oe2b.json',encoding='utf8')); eb=json.load(open('oe2b/esperado_local/exploratorio_oe2b.json',encoding='utf8'))
print('EXPLORATORIO: con/sin zoom', ea['n_con_zoom'], ea['n_sin_zoom'], '|', eb['n_con_zoom'], eb['n_sin_zoom'])
print('intentos descartados por Locust (peticiones, errores):', [(d['peticiones'],d['errores']) for d in ea['intentos_descartados_locust']], 'local:', [(d['peticiones'],d['errores']) for d in eb['intentos_descartados_locust']])
print('error acumulado:', ea['error_acumulado'], '|', eb['error_acumulado'])

## Conclusión de reproducibilidad

In [ ]:
import json, os
def _hojas(x, p=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from _hojas(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from _hojas(v, p+'/'+str(i))
    else: yield p, x
def _verif(a, b):
    A = dict(_hojas(json.load(open(a, encoding='utf8')))); B = dict(_hojas(json.load(open(b, encoding='utf8'))))
    mx = 0.0; ng = 0; malas = []
    for k in A:
        if k in B and isinstance(A[k], (int, float)) and isinstance(B[k], (int, float)) and not isinstance(A[k], bool) and not isinstance(B[k], bool):
            d = abs(A[k]-B[k]); mx = max(mx, d)
            if d > 1e-9 and d/max(abs(B[k]), 1e-300) > 1e-9: ng += 1; malas.append((k, A[k], B[k]))
        elif k in B and A[k] != B[k]: ng += 1; malas.append((k, A[k], B[k]))
    return set(A) == set(B), len(A), mx, ng, malas
CHECKS = [('OE2-A resultados_oe2a.json', 'oe2a/salida_colab/resultados_oe2a.json', 'oe2a/esperado_resultados_oe2a_local.json'), ('OE2-B pre-registrado resultados_oe2b.json', 'oe2b/salida_colab/preregistrado/resultados_oe2b.json', 'oe2b/esperado_local/resultados_oe2b.json'), ('OE2-B exploratorio exploratorio_oe2b.json', 'oe2b/salida_colab/exploratorio/exploratorio_oe2b.json', 'oe2b/esperado_local/exploratorio_oe2b.json')]
HTTP = ['oe2b']
print('CONCLUSION DE REPRODUCIBILIDAD')
todo_ok = True
dep_version = False
for etiqueta, a, b in CHECKS:
    mismas, n, mx, ng, malas = _verif(a, b)
    solo_wilcoxon = ng > 0 and all(k.endswith('/p_wilcoxon') for k, _, _ in malas)
    ok = mismas and ng == 0; todo_ok &= ok or (mismas and solo_wilcoxon); dep_version |= (not ok) and mismas and solo_wilcoxon
    veredicto = 'COINCIDE' if ok else ('COINCIDE salvo p_wilcoxon (aproximación normal que depende de la versión de scipy)' if mismas and solo_wilcoxon else 'NO COINCIDE')
    print(f'  {etiqueta}: misma estructura {mismas} | hojas {n} | diferencia absoluta máxima {mx:.3e} | hojas con diferencia > 1e-9: {ng} -> {veredicto}')
    for k, x, y in malas[:10]: print(f'     hoja distinta: {k} | Colab {x!r} | local {y!r}')
for k in HTTP:
    r = json.load(open(f'http/out/resumen_{k}.json'))
    ok = r['corridas_validas_finales_recalculadas'] == r['corridas_validas_finales_con_todos_los_campos_iguales'] and max(r['diferencia_abs_max_por_campo'].values()) == 0
    todo_ok &= ok
    print(f"  {k}: indicadores por corrida recalculados desde las peticiones: {r['corridas_validas_finales_recalculadas']} corridas válidas, {r['corridas_validas_finales_con_todos_los_campos_iguales']} idénticas -> {'COINCIDE' if ok else 'NO COINCIDE'}")
if todo_ok and dep_version: print('Resultado: el análisis se reproduce con los datos oficiales; todas las hojas coinciden dentro de 1e-9 salvo los p de Wilcoxon listados arriba, que dependen de la versión de scipy instalada (scipy', __import__('scipy').__version__, 'en este entorno). Las demás cantidades, incluido el ajuste de Holm calculado con p_t, coinciden.')
else: print('Resultado:', 'el análisis se reproduce con los datos oficiales y coincide con el resultado local dentro de la tolerancia de 1e-9 (las diferencias de coma flotante se informan arriba).' if todo_ok else 'HAY DIFERENCIAS: revise las líneas marcadas como NO COINCIDE antes de concluir.')
print('Alcance: se vuelve a ejecutar el análisis estadístico sobre los datos de entrada; no se repiten las campañas ni se regeneran los ciclos o las corridas, y la reproducción no implica que las condiciones de adquisición fueran controladas.')